**Summmary:**

In [1]:
import gzip
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

We dont have rating text here or the geomap so we will need to transform the JSON to get some of that but it looks like the **GOLD** is in the meta data from the JSON file since this is the **ONLY CSV**

The GMAP_ID is the ID of the business itself so we could join tables pretty easy if we needed to on that but would need to **Group by the business** on some bound so we dont repeat the business and increase the size of the table.

Did the state column load in properly?

In [2]:
#Understand more in depth, Gemine helped heaviy
def load_json_gzp(path: str) -> pd.DataFrame:
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

# Create 8 regions
def add_va_regions(
    df: pd.DataFrame, lat_col="latitude", lon_col="longitude", n_lat_bins=2, n_lon_bins=4, method="equal_width",):
    d = df.copy()

    # coords. Remove no coords
    d = d.dropna(subset=[lat_col, lon_col])
    d[lat_col] = pd.to_numeric(d[lat_col], errors="coerce")
    d[lon_col] = pd.to_numeric(d[lon_col], errors="coerce")
    #Why listed twice?
    d = d.dropna(subset=[lat_col, lon_col])

    # Gemini helped. qcut?
    if method == "quantile":
        d["lat_bin"] = pd.qcut(d[lat_col], q=n_lat_bins, labels=False, duplicates="drop")
        d["lon_bin"] = pd.qcut(d[lon_col], q=n_lon_bins, labels=False, duplicates="drop")
    else:
        #4 vert cols
        #look more into linspace
        lat_edges = np.linspace(d[lat_col].min(), d[lat_col].max(), n_lat_bins + 1)
        lon_edges = np.linspace(d[lon_col].min(), d[lon_col].max(), n_lon_bins + 1)
        d["lat_bin"] = pd.cut(d[lat_col], lat_edges, labels=False, include_lowest=True)
        d["lon_bin"] = pd.cut(d[lon_col], lon_edges, labels=False, include_lowest=True)

    d["lat_bin"] = d["lat_bin"].astype("Int64")
    d["lon_bin"] = d["lon_bin"].astype("Int64")

    #Look more into math behind this
    d["region_id"] = (d["lat_bin"] * n_lon_bins + d["lon_bin"]).astype("Int64")

    # Make readable
    lon_names = ["West", "MidWest", "MidEast", "East"]
    d["region_name"] = d.apply(lambda r: (f"{'South' if r.lat_bin == 0 else 'North'}-"f"{lon_names[int(r.lon_bin)]}") if pd.notna(r.region_id) else pd.NA, axis=1,)
    return d


meta_path = "data/meta-Virginia.json.gz"
review_path = "data/review-Virginia.json.gz"

df_meta = load_json_gzp(meta_path)
df_review = load_json_gzp(review_path)

#df with 8 regions
df_meta = add_va_regions(df_meta, method="equal_width")

#dict with id linked to region
region_lookup = (df_meta[["gmap_id", "region_id", "region_name"]].drop_duplicates("gmap_id"))

#uses gmap id to find loc of review
df_review = df_review.merge(region_lookup, on="gmap_id", how="left")

#Calculate statistics
business_summary = (
    df_meta.groupby(["region_id", "region_name"]).agg(
        n_businesses=("gmap_id", "nunique"),
        avg_rating=("avg_rating", "mean"),
        median_rating=("avg_rating", "median"),
        avg_num_reviews=("num_of_reviews", "mean"),
    ).reset_index().sort_values("region_id"))

# review length
df_review["review_length"] = (
    df_review["text"].fillna("").astype(str).str.len()
    if "text" in df_review.columns else np.nan #look more into nan because code kept breaking
)

#How to make easier for power BI?
review_summary = (
    df_review.groupby(["region_id", "region_name"]).agg(
        n_reviews=("gmap_id", "size"),
        avg_review_rating=("rating", "mean") if "rating" in df_review.columns else ("gmap_id", "size"),
        avg_review_length=("review_length", "mean"),
    ).reset_index().sort_values("region_id"))

print("Business summary by region:")
print(business_summary)

print("\nReview summary by region:")
print(review_summary)



Business summary by region:
   region_id region_name  n_businesses  avg_rating  median_rating  \
0          0  South-West             3    4.366667            4.2   
1          3  South-East            11    4.154545            4.1   
2          4  North-West        119017    4.273322            4.4   

   avg_num_reviews  
0        58.333333  
1       183.181818  
2       133.343484  

Review summary by region:
   region_id region_name  n_reviews  avg_review_rating  avg_review_length
0          0  South-West        175           4.062857          27.057143
1          3  South-East       2015           3.946898          95.353846
2          4  North-West   15955748           4.264254          99.268954


In [3]:
review_summary.shape[0]

3

Tablaeu
Power Bi
add markdowns to explain each piece of code
push to branch